<a href="https://colab.research.google.com/github/AshokGit544/Enterprise_LoRA_Domain_Adaptation_Platform/blob/main/Enterprise_LoRA_Domain__Adaptation__Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas numpy scikit-learn

In [2]:
from pathlib import Path
from datetime import datetime
import random
import json
import re

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [3]:
BASE_PATH = Path('/content/enterprise_lora_domain_adaptation_platform')
RAW_PATH = BASE_PATH / 'data' / 'raw_documents'
PROCESSED_PATH = BASE_PATH / 'data' / 'processed_documents'
FEATURE_PATH = BASE_PATH / 'data' / 'features'
OUTPUT_PATH = BASE_PATH / 'outputs'
CONFIG_PATH = BASE_PATH / 'config'

for p in [RAW_PATH, PROCESSED_PATH, FEATURE_PATH, OUTPUT_PATH, CONFIG_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/enterprise_lora_domain_adaptation_platform


In [4]:
project_schema = {
    'document_types': [
        'quality_report',
        'supplier_invoice',
        'maintenance_record',
        'engineering_change_notice'
    ],
    'domains': [
        'general_manufacturing',
        'specialized_manufacturing_domain'
    ],
    'target_use_case': 'domain_adapted_document_classification',
    'lora_concept': {
        'base_model': 'frozen_text_representation',
        'adapter_type': 'low_rank_trainable_adapter',
        'goal': 'adapt to enterprise-specific document patterns efficiently'
    }
}

with open(CONFIG_PATH / 'project_schema.json', 'w') as f:
    json.dump(project_schema, f, indent=2)

print(json.dumps(project_schema, indent=2))

{
  "document_types": [
    "quality_report",
    "supplier_invoice",
    "maintenance_record",
    "engineering_change_notice"
  ],
  "domains": [
    "general_manufacturing",
    "specialized_manufacturing_domain"
  ],
  "target_use_case": "domain_adapted_document_classification",
  "lora_concept": {
    "base_model": "frozen_text_representation",
    "adapter_type": "low_rank_trainable_adapter",
    "goal": "adapt to enterprise-specific document patterns efficiently"
  }
}


In [5]:
documents = []

supplier_names = ['Magna', 'Bosch', 'Denso', 'Lear', 'Aptiv']
plants = ['PLT100', 'PLT200', 'PLT300', 'PLT400']
issue_types = ['weld defect', 'paint inconsistency', 'sensor failure', 'material shortage', 'assembly variance']
equipment_ids = ['EQP1001', 'EQP1002', 'EQP1003', 'EQP1004']

for i in range(1, 801):
    doc_type = random.choice(project_schema['document_types'])
    plant = random.choice(plants)
    supplier = random.choice(supplier_names)
    issue = random.choice(issue_types)
    equipment = random.choice(equipment_ids)
    doc_date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    amount = max(1000.0, round(float(np.random.normal(12500, 3500)), 2))
    change_order_id = f"ECO-{random.randint(10000,99999)}"

    # Create general documents
    if i <= 500:
        domain = 'general_manufacturing'

        if doc_type == 'quality_report':
            text = (
                f"Document ID DOC{i:05d}. Plant {plant}. Quality inspection found {issue} on equipment {equipment}. "
                f"Document date {doc_date}. Supplier reference {supplier}. The report explains defect review and corrective action."
            )
        elif doc_type == 'supplier_invoice':
            text = (
                f"Document ID DOC{i:05d}. Supplier invoice from {supplier} for plant {plant}. Invoice amount ${amount}. "
                f"Document date {doc_date}. Material issue related to {issue}. The invoice includes purchasing and cost notes."
            )
        elif doc_type == 'maintenance_record':
            text = (
                f"Document ID DOC{i:05d}. Maintenance record for equipment {equipment} at plant {plant}. "
                f"Reported issue {issue}. Service date {doc_date}. Vendor {supplier}. The note explains service follow-up."
            )
        else:
            text = (
                f"Document ID DOC{i:05d}. Engineering change notice {change_order_id} for plant {plant}. "
                f"Change related to {issue}. Effective date {doc_date}. Supplier {supplier}. "
                f"The notice explains production change impact."
            )

    # Create domain-specific documents with extra vocabulary/patterns
    else:
        domain = 'specialized_manufacturing_domain'

        if doc_type == 'quality_report':
            text = (
                f"Document ID DOC{i:05d}. Plant {plant}. Quality deviation found {issue} on equipment {equipment}. "
                f"Supplier source {supplier}. Document date {doc_date}. "
                f"PFMEA review, torque validation, weld traceability, station-level containment, and defect escape analysis were recorded."
            )
        elif doc_type == 'supplier_invoice':
            text = (
                f"Document ID DOC{i:05d}. Supplier commercial document from {supplier} for plant {plant}. "
                f"Invoice amount ${amount}. Document date {doc_date}. "
                f"PPAP dependency, batch-level reconciliation, inbound material hold, and cost variance review were noted for {issue}."
            )
        elif doc_type == 'maintenance_record':
            text = (
                f"Document ID DOC{i:05d}. Equipment service log for {equipment} at plant {plant}. "
                f"Reported condition {issue}. Service date {doc_date}. Vendor {supplier}. "
                f"Station downtime, preventive cycle deviation, root cause inspection, and line recovery notes were captured."
            )
        else:
            text = (
                f"Document ID DOC{i:05d}. Engineering release notice {change_order_id} for plant {plant}. "
                f"Change related to {issue}. Effective date {doc_date}. Supplier {supplier}. "
                f"APQP alignment, process sheet revision, fixture adjustment, and manufacturing implementation checkpoints were documented."
            )

    documents.append({
        'document_id': f'DOC{i:05d}',
        'domain': domain,
        'document_type': doc_type,
        'document_text': text
    })

documents_df = pd.DataFrame(documents)
documents_df.to_csv(RAW_PATH / 'enterprise_documents.csv', index=False)

print(documents_df.head())
print(documents_df['domain'].value_counts())

  document_id                 domain   document_type  \
0    DOC00001  general_manufacturing  quality_report   
1    DOC00002  general_manufacturing  quality_report   
2    DOC00003  general_manufacturing  quality_report   
3    DOC00004  general_manufacturing  quality_report   
4    DOC00005  general_manufacturing  quality_report   

                                       document_text  
0  Document ID DOC00001. Plant PLT100. Quality in...  
1  Document ID DOC00002. Plant PLT400. Quality in...  
2  Document ID DOC00003. Plant PLT200. Quality in...  
3  Document ID DOC00004. Plant PLT200. Quality in...  
4  Document ID DOC00005. Plant PLT100. Quality in...  
domain
general_manufacturing               500
specialized_manufacturing_domain    300
Name: count, dtype: int64


In [6]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

documents_df = pd.read_csv(RAW_PATH / 'enterprise_documents.csv')
documents_df['normalized_text'] = documents_df['document_text'].apply(normalize_text)

documents_df.to_csv(PROCESSED_PATH / 'processed_documents.csv', index=False)

print(documents_df.head())

  document_id                 domain   document_type  \
0    DOC00001  general_manufacturing  quality_report   
1    DOC00002  general_manufacturing  quality_report   
2    DOC00003  general_manufacturing  quality_report   
3    DOC00004  general_manufacturing  quality_report   
4    DOC00005  general_manufacturing  quality_report   

                                       document_text  \
0  Document ID DOC00001. Plant PLT100. Quality in...   
1  Document ID DOC00002. Plant PLT400. Quality in...   
2  Document ID DOC00003. Plant PLT200. Quality in...   
3  Document ID DOC00004. Plant PLT200. Quality in...   
4  Document ID DOC00005. Plant PLT100. Quality in...   

                                     normalized_text  
0  document id doc00001. plant plt100. quality in...  
1  document id doc00002. plant plt400. quality in...  
2  document id doc00003. plant plt200. quality in...  
3  document id doc00004. plant plt200. quality in...  
4  document id doc00005. plant plt100. quality in..

In [7]:
general_df = documents_df[documents_df['domain'] == 'general_manufacturing'].copy()
specialized_df = documents_df[documents_df['domain'] == 'specialized_manufacturing_domain'].copy()

general_train_df, general_test_df = train_test_split(
    general_df,
    test_size=0.2,
    random_state=42,
    stratify=general_df['document_type']
)

specialized_train_df, specialized_test_df = train_test_split(
    specialized_df,
    test_size=0.2,
    random_state=42,
    stratify=specialized_df['document_type']
)

print('General train:', len(general_train_df))
print('General test:', len(general_test_df))
print('Specialized train:', len(specialized_train_df))
print('Specialized test:', len(specialized_test_df))

General train: 400
General test: 100
Specialized train: 240
Specialized test: 60


In [8]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=300)

X_general_train = vectorizer.fit_transform(general_train_df['normalized_text']).toarray()
X_general_test = vectorizer.transform(general_test_df['normalized_text']).toarray()

X_specialized_train = vectorizer.transform(specialized_train_df['normalized_text']).toarray()
X_specialized_test = vectorizer.transform(specialized_test_df['normalized_text']).toarray()

y_general_train = general_train_df['document_type'].values
y_general_test = general_test_df['document_type'].values

y_specialized_train = specialized_train_df['document_type'].values
y_specialized_test = specialized_test_df['document_type'].values

print('Base feature shape:', X_general_train.shape)

Base feature shape: (400, 300)


In [9]:
base_clf = LogisticRegression(max_iter=2000, random_state=42)
base_clf.fit(X_general_train, y_general_train)

general_preds = base_clf.predict(X_general_test)
specialized_base_preds = base_clf.predict(X_specialized_test)

general_accuracy = accuracy_score(y_general_test, general_preds)
specialized_base_accuracy = accuracy_score(y_specialized_test, specialized_base_preds)

print('Base model accuracy on general test data:', round(float(general_accuracy), 4))
print('Base model accuracy on specialized domain test data:', round(float(specialized_base_accuracy), 4))

print('\nBase model report on specialized domain:')
print(classification_report(y_specialized_test, specialized_base_preds))

Base model accuracy on general test data: 1.0
Base model accuracy on specialized domain test data: 1.0

Base model report on specialized domain:
                           precision    recall  f1-score   support

engineering_change_notice       1.00      1.00      1.00        15
       maintenance_record       1.00      1.00      1.00        15
           quality_report       1.00      1.00      1.00        15
         supplier_invoice       1.00      1.00      1.00        15

                 accuracy                           1.00        60
                macro avg       1.00      1.00      1.00        60
             weighted avg       1.00      1.00      1.00        60



In [10]:
rank = 16  # low-rank size
input_dim = X_general_train.shape[1]

A = np.random.normal(0, 0.02, size=(input_dim, rank))
B = np.random.normal(0, 0.02, size=(rank, input_dim))

def apply_low_rank_adapter(X, A, B, alpha=0.5):
    # LoRA-style idea: original features + small low-rank adaptation
    delta = X @ A @ B
    return X + alpha * delta

X_specialized_train_adapted = apply_low_rank_adapter(X_specialized_train, A, B, alpha=0.5)
X_specialized_test_adapted = apply_low_rank_adapter(X_specialized_test, A, B, alpha=0.5)

print('Adapted train feature shape:', X_specialized_train_adapted.shape)

Adapted train feature shape: (240, 300)


In [11]:
adapter_clf = LogisticRegression(max_iter=2000, random_state=42)
adapter_clf.fit(X_specialized_train_adapted, y_specialized_train)

specialized_adapter_preds = adapter_clf.predict(X_specialized_test_adapted)
specialized_adapter_accuracy = accuracy_score(y_specialized_test, specialized_adapter_preds)

print('Adapter model accuracy on specialized domain test data:', round(float(specialized_adapter_accuracy), 4))

print('\nAdapter model report on specialized domain:')
print(classification_report(y_specialized_test, specialized_adapter_preds))

Adapter model accuracy on specialized domain test data: 1.0

Adapter model report on specialized domain:
                           precision    recall  f1-score   support

engineering_change_notice       1.00      1.00      1.00        15
       maintenance_record       1.00      1.00      1.00        15
           quality_report       1.00      1.00      1.00        15
         supplier_invoice       1.00      1.00      1.00        15

                 accuracy                           1.00        60
                macro avg       1.00      1.00      1.00        60
             weighted avg       1.00      1.00      1.00        60



In [12]:
comparison_df = pd.DataFrame({
    'model_version': ['base_model', 'lora_style_adapter_model'],
    'general_domain_accuracy': [round(float(general_accuracy), 4), np.nan],
    'specialized_domain_accuracy': [
        round(float(specialized_base_accuracy), 4),
        round(float(specialized_adapter_accuracy), 4)
    ]
})

comparison_df.to_csv(OUTPUT_PATH / 'model_comparison.csv', index=False)
comparison_df

,model_version,general_domain_accuracy,specialized_domain_accuracy
0,base_model,1.0,1.0
1,lora_style_adapter_model,NaN,1.0


In [13]:
example_results_df = specialized_test_df[['document_id', 'domain', 'document_type', 'document_text']].copy().reset_index(drop=True)
example_results_df['base_prediction'] = specialized_base_preds
example_results_df['adapter_prediction'] = specialized_adapter_preds

example_results_df.to_csv(OUTPUT_PATH / 'example_predictions.csv', index=False)

example_results_df.head(10)

,document_id,domain,document_type,document_text,base_prediction,adapter_prediction
0,DOC00797,specialized_manufacturing_domain,quality_report,Document ID DOC00797. Plant PLT100. Quality de...,quality_report,quality_report
1,DOC00723,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00723. Engineering release noti...,engineering_change_notice,engineering_change_notice
2,DOC00792,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00792. Engineering release noti...,engineering_change_notice,engineering_change_notice
3,DOC00790,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00790. Engineering release noti...,engineering_change_notice,engineering_change_notice
4,DOC00765,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00765. Engineering release noti...,engineering_change_notice,engineering_change_notice
5,DOC00747,specialized_manufacturing_domain,quality_report,Document ID DOC00747. Plant PLT100. Quality de...,quality_report,quality_report
6,DOC00731,specialized_manufacturing_domain,supplier_invoice,Document ID DOC00731. Supplier commercial docu...,supplier_invoice,supplier_invoice
7,DOC00602,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00602. Engineering release noti...,engineering_change_notice,engineering_change_notice
8,DOC00669,specialized_manufacturing_domain,supplier_invoice,Document ID DOC00669. Supplier commercial docu...,supplier_invoice,supplier_invoice
9,DOC00777,specialized_manufacturing_domain,supplier_invoice,Document ID DOC00777. Supplier commercial docu...,supplier_invoice,supplier_invoice


In [14]:
base_parameter_count = input_dim * len(np.unique(y_general_train))
adapter_parameter_count_estimate = (input_dim * rank) + (rank * input_dim)

parameter_summary = {
    'input_dimension': int(input_dim),
    'low_rank_r': int(rank),
    'estimated_base_parameter_scale': int(base_parameter_count),
    'estimated_adapter_parameter_scale': int(adapter_parameter_count_estimate),
    'concept_note': 'The adapter updates a smaller low-rank structure instead of retraining a full large backbone.'
}

with open(OUTPUT_PATH / 'parameter_summary.json', 'w') as f:
    json.dump(parameter_summary, f, indent=2)

print(json.dumps(parameter_summary, indent=2))

{
  "input_dimension": 300,
  "low_rank_r": 16,
  "estimated_base_parameter_scale": 1200,
  "estimated_adapter_parameter_scale": 9600,
  "concept_note": "The adapter updates a smaller low-rank structure instead of retraining a full large backbone."
}


In [15]:
project_summary = {
    'generated_at': datetime.now().isoformat(),
    'total_documents': int(len(documents_df)),
    'general_domain_documents': int(len(general_df)),
    'specialized_domain_documents': int(len(specialized_df)),
    'base_model_general_accuracy': round(float(general_accuracy), 4),
    'base_model_specialized_accuracy': round(float(specialized_base_accuracy), 4),
    'adapter_model_specialized_accuracy': round(float(specialized_adapter_accuracy), 4),
    'improvement_over_base_on_specialized_domain': round(float(specialized_adapter_accuracy - specialized_base_accuracy), 4),
    'low_rank_r': int(rank)
}

with open(OUTPUT_PATH / 'project_summary.json', 'w') as f:
    json.dump(project_summary, f, indent=2)

print(json.dumps(project_summary, indent=2))

{
  "generated_at": "2026-03-17T20:49:03.114927",
  "total_documents": 800,
  "general_domain_documents": 500,
  "specialized_domain_documents": 300,
  "base_model_general_accuracy": 1.0,
  "base_model_specialized_accuracy": 1.0,
  "adapter_model_specialized_accuracy": 1.0,
  "improvement_over_base_on_specialized_domain": 0.0,
  "low_rank_r": 16
}


In [16]:
print('--- MODEL COMPARISON ---')
display(comparison_df)

print('\n--- SAMPLE SPECIALIZED DOMAIN PREDICTIONS ---')
display(example_results_df.head(10))

print('\n--- PROJECT SUMMARY ---')
print(json.dumps(project_summary, indent=2))

--- MODEL COMPARISON ---


,model_version,general_domain_accuracy,specialized_domain_accuracy
0,base_model,1.0,1.0
1,lora_style_adapter_model,NaN,1.0



--- SAMPLE SPECIALIZED DOMAIN PREDICTIONS ---


,document_id,domain,document_type,document_text,base_prediction,adapter_prediction
0,DOC00797,specialized_manufacturing_domain,quality_report,Document ID DOC00797. Plant PLT100. Quality de...,quality_report,quality_report
1,DOC00723,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00723. Engineering release noti...,engineering_change_notice,engineering_change_notice
2,DOC00792,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00792. Engineering release noti...,engineering_change_notice,engineering_change_notice
3,DOC00790,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00790. Engineering release noti...,engineering_change_notice,engineering_change_notice
4,DOC00765,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00765. Engineering release noti...,engineering_change_notice,engineering_change_notice
5,DOC00747,specialized_manufacturing_domain,quality_report,Document ID DOC00747. Plant PLT100. Quality de...,quality_report,quality_report
6,DOC00731,specialized_manufacturing_domain,supplier_invoice,Document ID DOC00731. Supplier commercial docu...,supplier_invoice,supplier_invoice
7,DOC00602,specialized_manufacturing_domain,engineering_change_notice,Document ID DOC00602. Engineering release noti...,engineering_change_notice,engineering_change_notice
8,DOC00669,specialized_manufacturing_domain,supplier_invoice,Document ID DOC00669. Supplier commercial docu...,supplier_invoice,supplier_invoice
9,DOC00777,specialized_manufacturing_domain,supplier_invoice,Document ID DOC00777. Supplier commercial docu...,supplier_invoice,supplier_invoice



--- PROJECT SUMMARY ---
{
  "generated_at": "2026-03-17T20:49:03.114927",
  "total_documents": 800,
  "general_domain_documents": 500,
  "specialized_domain_documents": 300,
  "base_model_general_accuracy": 1.0,
  "base_model_specialized_accuracy": 1.0,
  "adapter_model_specialized_accuracy": 1.0,
  "improvement_over_base_on_specialized_domain": 0.0,
  "low_rank_r": 16
}
